In [ ]:
# Install required packages
%pip install --quiet -U langchain_core langgraph langchain_openai

## Just LLM without any tools

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

model = ChatOpenAI(model="gpt-4o")

In [ ]:
model.invoke([{"role": "user", "content": "Good morning!"}])

## Creation of simple agent

In [ ]:
def check_weather(location: str) -> str:
    '''Return the weather forecast for the specified location.'''
    return f"It's always sunny in {location}"


weather_agent = create_agent(
    model=model,
    tools=[check_weather],
    system_prompt="You are a helpful assistant",
)


In [ ]:
from IPython.display import Image, display
display(Image(weather_agent.get_graph().draw_mermaid_png()))

In [ ]:
inputs = {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
for chunk in weather_agent.stream(inputs, stream_mode="updates"):
    print(chunk)

In [ ]:
response = weather_agent.invoke(inputs)

In [ ]:
#response['messages']
for m in response['messages']:
#for m in result['messages']: # Print last message only
    m.pretty_print()

## Warehouse Management System

In [ ]:
# Robot Controller agent: executes concrete tasks
from langchain_core.messages import HumanMessage
def robot_controller_tool(task: str)-> str:
    """Executes the given task."""
    print('Robot Controller executing task', task)
    return f"Executing task: {task}"

robot_controller = create_agent(
    model=model,
    tools=[robot_controller_tool],
    system_prompt="You are the Robot Controller. Execute assigned tasks precisely.",
    name="robot_controller",
)



In [ ]:
# Zone Manager agent: delegates tasks to Robot Controller
def zone_manager_tool(task: str) -> str:
    """Delegates task to Robot Controller agent."""
    print('Zone Manager delegating task', task)
    # Use correct message format for LangGraph agents
    return robot_controller.invoke({"messages": [HumanMessage(content=task)]})

zone_manager = create_agent(
    model=model,
    tools=[zone_manager_tool],
    system_prompt="You are the Zone Manager. Delegate given tasks to Robot Controllers.",
    name="zone_manager",
)

In [ ]:
# Warehouse Manager agent: delegates tasks to Zone Manager
def warehouse_manager_tool(task: str) -> str:
    """Delegates task to Zone Manager agent."""
    print('Warehouse Manager delegating task', task)
    # Use correct message format for LangGraph agents
    return zone_manager.invoke({"messages": [HumanMessage(content=task)]})

warehouse_manager = create_agent(
    model=model,
    tools=[warehouse_manager_tool],
    system_prompt="You are the Warehouse Manager. Given the task by the user, Delegate tasks to Zone Managers.",
    name="warehouse_manager",
)

In [ ]:
# Begin hierarchical task delegation from Warehouse Manager down to Robot Controller
task = "Move all packages from Zone A to Zone B."

In [ ]:
from IPython.display import Image, display
display(Image(warehouse_manager.get_graph().draw_mermaid_png()))

In [ ]:
# response = warehouse_manager.stream({"role": "user", "content": f"Delegate task: {task}"}, stream_mode="updates") 
# for chunk in response:
#     print(chunk) 


In [ ]:
inputs = {"messages": [{"role": "user", "content": f"Delegate task: {task}"}]}
response = warehouse_manager.invoke(inputs)
for m in response['messages']:
    m.pretty_print()

In [ ]:
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, ToolMessage

for msg, metadata in warehouse_manager.stream(inputs, stream_mode="messages"):
    node_name = metadata.get("langgraph_node")
    if node_name == "model":
        # Print LLM messages
        if msg.content:
            print(f"LLM: {msg.content}")
        # If the LLM generates tool calls, you might want to print them too
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tool_call in msg.tool_calls:
                print(f"LLM calls tool: {tool_call['name']} with args: {tool_call['args']}")
    elif node_name == "tools":
        # Print tool messages and their names
        if isinstance(msg, ToolMessage) and msg.name:
            print(f"*** Called tool: {msg.name} ***")
            print(f"Tool output: {msg.content}")
    else:
        print(f"Unexpected node name: {node_name}")

## Exercise 1:
1. Create two agents - Research Agent and News Agent
2. Create one supervisor agent.
3. Connect both agents and test the flow
